# Bead Annotation Walkthrough

This notebook builds the annotation tool step by step before it becomes a reusable script.

**What we're doing:**
- loading a `.czi` microscopy image,
- opening it in napari (an interactive image viewer),
- clicking on every bead to mark its center,
- then saving those coordinates to a CSV.

## Step 0: Imports

- `pathlib.Path` — for handling file paths in a way that works on Windows and Mac/Linux alike
- `numpy` — the image, once loaded, is just a 2D grid of numbers (pixel intensities); numpy is
  how Python represents that
- `pandas` — to build and save the table of coordinates
- `BioImage` — reads the `.czi` file (from `bioio`, the actively-maintained successor to the
  now-discontinued `aicsimageio` package)
- `napari` — the interactive viewer

In [35]:
from pathlib import Path

import numpy as np
import pandas as pd
from bioio import BioImage
import napari

## Step 1: Enable napari inside Jupyter

Napari opens as its own interactive window, built with a GUI toolkit called **Qt** — it does not
render inline in the notebook like a matplotlib plot does. The `%gui qt` line below tells Jupyter
to run Qt's event loop alongside the notebook's own loop, so the napari window stays responsive
*while you keep running notebook cells*. This only needs to be run once per notebook session.

In [36]:
%gui qt

## Step 2: Load one image

Set `image_path` to one of your `.czi` files. `BioImage` reads the file and lets us pull out a
single 2D plane. `.czi` files can contain multiple channels, Z-planes (depth slices), scenes, and
timepoints all in one file — `get_image_data("YX", ...)` says "give me a flat 2D image (Y rows,
X columns)" and the `C=0, T=0, Z=0` arguments pick the *first* channel/time/Z-slice.

**Run the `print` lines below first** — they show you the image's actual dimensions and channel
names, so you can confirm channel 0 really is your SMI-31 channel before annotating anything.

In [37]:
image_path = Path("../data/raw/control/NI240119_SMI31-488_20x_NT_01.czi")

img = BioImage(str(image_path))
print("Dimensions:", img.dims)          # shows what T, C, Z, Y, X sizes actually are
print("Channel names:", img.channel_names)

image = img.get_image_data("YX", C=1, T=0, Z=0)
print("2D image shape:", image.shape)

Dimensions: <Dimensions [T: 1, C: 2, Z: 1, Y: 1024, X: 1024]>
Channel names: [np.str_('Ch1-T1'), np.str_('Ch2-T2')]
2D image shape: (1024, 1024)


If `channel_names` shows more than one channel and SMI-31 isn't first, change `C=0` above to
the correct index and re-run the cell before continuing.

## Step 3: Open napari and view the image

This opens the actual interactive window. Because of `%gui qt` above, this cell finishes running
immediately and the window stays open — you can keep using the notebook.

In [38]:
viewer = napari.Viewer(title=f"Annotating: {image_path.name}")
viewer.add_image(image, name="SMI-31", colormap="gray_r")
#Set color map to viridis or gray r

<Image layer 'SMI-31' at 0x2bed3328050>

## Step 4: Add a Points layer

A napari "layer" is one visual element in the viewer — the image itself is one layer, and now
we're adding a second layer just for your clicks. Points you add here appear as circles overlaid
on the image, without modifying the underlying image data at all.

In [39]:
points_layer = viewer.add_points(
    name="annotations",
    ndim=2,
    size=5,
    face_color="yellow",
    border_color="black",
)

points_layer.mode = 'add'

## Step 5: Go click on beads

Switch to the napari window. In the left panel, make sure the **"annotations"** layer is
selected (highlighted). Select the point tool (the dot icon in the top toolbar) if it isn't
already active, then **click directly on each bead** in the image to mark its center.

Made a mistake? Switch to the select tool (arrow icon), click the point to select it, and press
Backspace/Delete.

**Don't close the window** — just come back to this notebook when you've marked every bead and
run the next cell.

## Step 6: Pull out what you clicked

`points_layer.data` is a live array of every point you've placed, in `(row, col)` — i.e.
`(y, x)` — pixel coordinates. Run this any time; you can even run it, keep clicking more points,
and run it again later.

In [40]:
coords = points_layer.data
print(f"{len(coords)} beads marked so far")

df = pd.DataFrame(coords, columns=["y", "x"])
df.insert(0, "image", image_path.name)
df.head()

4 beads marked so far


,image,y,x
0,NI240119_SMI31-488_20x_NT_01.czi,577.397991,530.065686
1,NI240119_SMI31-488_20x_NT_01.czi,481.549357,341.951543
2,NI240119_SMI31-488_20x_NT_01.czi,471.695759,172.648815
3,NI240119_SMI31-488_20x_NT_01.czi,494.090300,156.524746


## Step 7: Save to CSV

This writes the annotations to `data/interim/annotations/`, matching the folder structure we
set up for the rest of the project.

In [41]:
output_dir = Path("../data/interim/annotations")
output_dir.mkdir(parents=True, exist_ok=True)

out_path = output_dir / f"{image_path.stem}_annotations.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} annotations to {out_path}")

Saved 4 annotations to ..\data\interim\annotations\NI240119_SMI31-488_20x_NT_01_annotations.csv


## Next image

To annotate another image: close this viewer window (`viewer.close()`, or just close it by hand),
then go back to **Step 2**, change `image_path` to the next file, and re-run Steps 2 through 7.

Once this feels repetitive rather than instructive, that's the natural point to turn Steps 2–7
into a single reusable function (or back into the standalone script) — but doing it by hand a few
times first is worth it for actually understanding what napari is doing under the hood.